# Train through the generic infrastructure layer

This notebook intentionally contains no optimizer, device, DDP, or epoch-loop code. All infrastructure behavior comes from YAML and `generic_trainer`. For multi-GPU execution, use the `torchrun` command in the final cell from a terminal.

In [1]:
# Log model size before allocating the model weights. The meta device lets this
# same cell inspect the 500M profile without consuming ~2 GiB of RAM.
from pathlib import Path
import json, sys, torch
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from generic_trainer.config import load_config
from models.housing_llm.factory import build_model
from generic_trainer.telemetry import model_size

config = load_config(ROOT / 'configs/smoke.yaml')
config['run']['output_dir'] = str(ROOT / 'artifacts/notebook-smoke')
model_config = {k: v for k, v in config['model'].items() if k != 'factory'}
with torch.device('meta'):
    size_report = model_size(build_model(model_config))
print(json.dumps(size_report, indent=2))

{
  "parameters": 201089,
  "trainable_parameters": 201089,
  "parameter_bytes": 804356,
  "buffer_bytes": 0,
  "model_bytes": 804356,
  "model_gib": 0.0007491149008274078
}


In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
ROOT

PosixPath('/Users/sathyasaravaiya/Documents/Codex/LLMDemo')

In [3]:
from generic_trainer.config import load_config

config = load_config(ROOT / 'configs/smoke.yaml', [
    'data.num_examples=64',
    'trainer.max_epochs=1',
])
config['run']['output_dir'] = str(ROOT / 'artifacts/notebook-smoke')
config

{'run': {'name': 'smoke',
  'seed': 7,
  'output_dir': '/Users/sathyasaravaiya/Documents/Codex/LLMDemo/artifacts/notebook-smoke'},
 'model': {'factory': 'models.housing_llm.factory:build_model',
  'vocab_size': 2048,
  'max_seq_len': 64,
  'd_model': 64,
  'n_layers': 2,
  'n_heads': 4,
  'd_ff': 128,
  'dropout': 0.0,
  'target_mean': 300000.0,
  'target_scale': 200000.0},
 'data': {'factory': 'models.housing_llm.factory:build_datasets',
  'source': {'type': 'synthetic',
   'path': None,
   'cache_dir': '.cache/public-data'},
  'num_examples': 64,
  'validation_fraction': 0.2,
  'feature_fields': ['state',
   'county',
   'bedrooms',
   'bathrooms',
   'living_area_sqft',
   'median_income',
   'population',
   'year_built'],
  'target_field': 'home_value',
  'tokenizer': {'vocab_size': 2048, 'max_length': 64}},
 'trainer': {'max_epochs': 1,
  'train_batch_size': 8,
  'eval_batch_size': 8,
  'gradient_accumulation_steps': 1,
  'learning_rate': 0.001,
  'weight_decay': 0.01,
  'warmup_

In [4]:
from generic_trainer.cli import run

metrics = run(config)
metrics

{"event": "startup", "device": "mps", "world_size": 1, "parameters": 201089, "train_examples": 51, "validation_examples": 13}
{"buffer_bytes": 0, "estimated_total_model_gib_across_ranks": 0.0007491149008274078, "event": "hardware_placement", "gpu_inventory": [], "model_bytes": 804356, "model_gib": 0.0007491149008274078, "parameter_bytes": 804356, "parameters": 201089, "rank_placements": [{"device": "mps", "device_type": "mps", "host": "Mac", "local_rank": 0, "physical_gpu_index": null, "rank": 0}], "trainable_parameters": 201089, "trainer_gpu_count": 0, "unused_gpu_indices": [], "world_size": 1}
{"epoch": 1, "event": "step", "examples_per_second": 53.06148173839682, "global_step": 1, "grad_norm": 12.825512886047363, "learning_rate": 0.001, "train_loss": 9.164395332336426}
{"event": "gpu_utilization", "global_step": 1, "gpu_inventory": [], "rank_samples": [{"device": "mps", "device_type": "mps", "host": "Mac", "local_rank": 0, "physical_gpu_index": null, "rank": 0}]}
{"epoch": 1, "event

{'train_loss': 7.791528701782227, 'val_loss': 7.231263160705566}

In [5]:
# Summarize exactly which ranks/GPUs the trainer used and which installed GPUs
# were deliberately idle. Point RUN_DIR at a torchrun artifact directory when
# presenting a multi-GPU run.
RUN_DIR = Path(config['run']['output_dir'])
events = [json.loads(line) for line in (RUN_DIR / 'metrics.jsonl').read_text().splitlines()]
placement = next(event for event in events if event['event'] == 'hardware_placement')
print(f"Model: {placement['trainable_parameters']:,} trainable parameters, {placement['model_gib']:.2f} GiB per replica")
print(f"Trainer ranks: {placement['world_size']}; GPUs used: {placement['trainer_gpu_count']}")
print(f"Unused GPU indices: {placement['unused_gpu_indices'] or 'none / no NVIDIA inventory'}")
placement['rank_placements']

Model: 201,089 trainable parameters, 0.00 GiB per replica
Trainer ranks: 1; GPUs used: 0
Unused GPU indices: none / no NVIDIA inventory


[{'device': 'mps',
  'device_type': 'mps',
  'host': 'Mac',
  'local_rank': 0,
  'physical_gpu_index': None,
  'rank': 0}]

In [6]:
# Leadership chart: utilization over time for every installed GPU. Labels show
# the assigned rank; unassigned devices are explicitly marked UNUSED.
import matplotlib.pyplot as plt
utilization_events = [event for event in events if event['event'] == 'gpu_utilization']
if not utilization_events or not utilization_events[0]['gpu_inventory']:
    print('No NVIDIA GPU samples in this run (expected for a CPU-only run).')
else:
    gpu_indices = [gpu['gpu_index'] for gpu in utilization_events[0]['gpu_inventory']]
    fig, (util_ax, memory_ax) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    for gpu_index in gpu_indices:
        samples = [next(g for g in event['gpu_inventory'] if g['gpu_index'] == gpu_index) for event in utilization_events]
        steps = [event['global_step'] for event in utilization_events]
        rank = samples[0]['rank']
        label = f'GPU {gpu_index} — rank {rank}' if rank is not None else f'GPU {gpu_index} — UNUSED'
        style = '-' if rank is not None else '--'
        util_ax.plot(steps, [s['utilization_percent'] for s in samples], style, label=label)
        memory_ax.plot(steps, [100*s['memory_used_mib']/s['memory_total_mib'] for s in samples], style, label=label)
    util_ax.set(ylabel='GPU utilization (%)', ylim=(0, 100), title='Trainer GPU utilization by rank')
    memory_ax.set(xlabel='Optimizer step', ylabel='GPU memory used (%)', ylim=(0, 100))
    for axis in (util_ax, memory_ax): axis.grid(alpha=.25); axis.legend(ncol=2)
    plt.tight_layout(); plt.show()

Matplotlib is building the font cache; this may take a moment.


No NVIDIA GPU samples in this run (expected for a CPU-only run).


For the real profile, launch outside Jupyter: `torchrun --standalone --nproc_per_node=4 -m generic_trainer.cli --config configs/train_500m.yaml`.